In [1]:
import numpy as np
import torchvision as thv

from torch import torch
from torch.utils.data import DataLoader

from common.utils import Utils
from common.nn_model import NNModel, NNTrainParams
from common.feed_fwd_nn import FeedFwdNN, FeedFwdNNParams

In [2]:
DS_MEAN : float = 0.1307
DS_STD  : float = 0.3081
    
train_dataset = thv.datasets.MNIST(
    root="./data"
    , train=True
    , download=True
    , transform=thv.transforms.Compose(
        [
            thv.transforms.ToTensor()
            , thv.transforms.Normalize(mean=DS_MEAN, std=DS_STD)
        ]
    )
)

val_dataset = thv.datasets.MNIST(
    root="./data"
    , train=False
    , download=True
    , transform=thv.transforms.Compose(
        [
            thv.transforms.ToTensor()
            , thv.transforms.Normalize(mean=DS_MEAN, std=DS_STD)
        ]
    )
)

val_loader = DataLoader(dataset=val_dataset, batch_size=len(val_dataset))
train_loader = DataLoader(dataset=train_dataset, batch_size=len(train_dataset))

print(len(val_dataset.data))
print(len(train_dataset.data))

10000
60000


In [4]:
n_epochs = 1
lrs = [0.001]
optims = ["adam"]

dropout_probs = [0.5]
hidden_dimss = [
    []
    # , [500]
    # , [1000]
    # , [500, 1000]
]

models = [
    model
        for model in [
            NNModel(
                device="cpu"
                , net=FeedFwdNN(
                    params=FeedFwdNNParams(
                        input_dim=28*28
                        , output_dim=10
                        , hidden_dims=hidden_dims
                        , dropout_prob=dropout_prob
                    )
                )
            )
                for dropout_prob in dropout_probs
                for hidden_dims in hidden_dimss
        ]
]

train_params = [
    train_param
        for train_param in [
            NNTrainParams(
                optim=optim
                , learning_rate=lr
                , n_epochs=n_epochs
                , val_loader=val_loader
                , train_loader=train_loader
            )
                for lr in lrs
                for optim in optims 
        ]
]

trains = {
    train_str: (model, train_idps)
        for model, (train_str, train_idps) in [
            (model, model.train(params=train_param))
                for model in models
                for train_param in train_params
        ]
}

FeedFwdNN{dims=[784, 10], dropout=0.5} x [epochs=1, lr=0.001, optim=adam]: 100%|██████████| 1/1 [00:05<00:00,  5.01s/it, error: 0.8223]


In [ ]:
top_model_names = [
    kvp[0] for kvp in sorted(
        list(trains.items())
        , key=lambda kvp: min(
            kvp[1][1]
            , key=lambda idp: idp.val_error
        ).val_error
    )[:5]
]

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=1
    , fig_size=(25, 20)
    , y_axis_label="Loss"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Losses"
    , x=[idp.iter_idx for idp in trains[top_model_names[0]][1]]
    , yss_legend=[[f"{loss_type} of {model_name}" for loss_type in ["training", "validation"]] for model_name in top_model_names]
    , yss=[[[model_idp.train_loss for model_idp in trains[model_name][1]], [model_idp.val_loss for model_idp in trains[model_name][1]]] for model_name in top_model_names]
)

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=1
    , fig_size=(25, 20)
    , y_axis_label="Error"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Errors"
    , x=[idp.iter_idx for idp in trains[top_model_names[0]][1]]
    , yss_legend=[[f"{loss_type} of {model_name}" for loss_type in ["training", "validation"]] for model_name in top_model_names]
    , yss=[[[model_idp.train_error for model_idp in trains[model_name][1]], [model_idp.val_error for model_idp in trains[model_name][1]]] for model_name in top_model_names]
)

In [ ]:
print(f"best model is {top_model_names[0]} which achieves validation error of {min(trains[top_model_names[0]][1], key=lambda idp: idp.val_error).val_error:.4f}")